In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score

pd.options.display.max_columns = 100
pd.options.display.max_rows = 100


df = pd.read_csv("../data/car_prices_kaggle.csv")

df = df.loc[~df.duplicated(subset=["ID"])].reset_index(drop=True).copy()
df = df.drop(columns=['Levy', 'ID', 'Doors', 'Leather interior', 'Wheel', 'Airbags']).copy()


In [14]:
df.head()

,Price,Manufacturer,Model,Prod. year,Category,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Color
0,13328,LEXUS,RX 450,2010,Jeep,Hybrid,3.5,186005 km,6.0,Automatic,4x4,Silver
1,16621,CHEVROLET,Equinox,2011,Jeep,Petrol,3,192000 km,6.0,Tiptronic,4x4,Black
2,8467,HONDA,FIT,2006,Hatchback,Petrol,1.3,200000 km,4.0,Variator,Front,Black
3,3607,FORD,Escape,2011,Jeep,Hybrid,2.5,168966 km,4.0,Automatic,4x4,White
4,11726,HONDA,FIT,2014,Hatchback,Petrol,1.3,91901 km,4.0,Automatic,Front,Silver


In [15]:
# Mapping

class Mapping(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):

        df = X.copy()
        
        color_map = {
            "Silver": "Silver",
            "Black": "Black",
            "White": "White",
            "Grey": "Gray",
            "Blue": "Blue",
            "Sky blue": "Blue",
            "Green": "Green",
            "Red": "Red",
            "Carnelian red": "Red",
            "Orange": "Orange",
            "Yellow": "Gold",
            "Brown": "Brown",
            "Golden": "Gold",
            "Beige": "Beige",
            "Purple": "Purple",
            "Pink": "Red",
        }
        
        BODY_TYPE_MAP = {
            "Jeep": "SUV",
            "Hatchback": "Hatchback",
            "Sedan": "Sedan",
            "Microbus": "Van",
            "Goods wagon": "Van",
            "Universal": "Wagon",
            "Coupe": "Coupe",
            "Minivan": "Van",
            "Cabriolet": "Convertible",
            "Limousine": "Sedan",
            "Pickup": "Pickup",
        }
        
        GEARBOX_TYPE_MAP = {
            "Automatic": "Automatic",
            "Tiptronic": "Automatic",
            "Variator": "Automatic",  # CVT
            "Manual": "Manual",
        }
        
        drive_wheels_map = {
            '4x4': 'AWD',
            'Front': 'FWD',
            'Rear': 'RWD'
        }
        
        fuel_type_map = {
            "Hybrid": "Hybrid",
            "Plug-in Hybrid": "Hybrid",
            "Petrol": "Petrol",
            "Diesel": "Diesel",
            "CNG": "Petrol",
            "LPG": "Petrol",
            "Hydrogen": "Electric",
        }

        df = df[df['Fuel type'] != 'Hydrogen'].copy()  # Very low hydrogen, remove vefore mapping
        df['Fuel type'] = df['Fuel type'].map(fuel_type_map)
        df['Drive wheels'] = df['Drive wheels'].map(drive_wheels_map)
        df['Gear box type'] = df['Gear box type'].map(GEARBOX_TYPE_MAP)
        df['Color'] = df['Color'].map(color_map)
        df['Category'] = df['Category'].map(BODY_TYPE_MAP)

        return df


In [16]:
class MakeCanonicalizer(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        supported_makes = [
            "Toyota", "Hyundai", "Nissan", "Kia", "Honda", "Lexus", "GMC",
            "Chevrolet", "Ford", "Tesla", "BMW", "Mercedes-Benz", "Mitsubishi",
            "Land Rover", "Jeep", "Dodge", "Ram", "Volkswagen", "Audi", "Mazda",
            "Infiniti", "Cadillac", "Subaru"
        ]

        make_lookup = {m.lower(): m for m in supported_makes}

        df["Manufacturer"] = df["Manufacturer"].astype(str).str.strip().str.lower()
        df = df[df["Manufacturer"].isin(make_lookup.keys())].copy()
        df["Manufacturer"] = df["Manufacturer"].map(make_lookup)

        make_counts = df["Manufacturer"].value_counts()
        rare_makes = make_counts[make_counts < self.min_count].index
        df["Manufacturer"] = df["Manufacturer"].replace(rare_makes, "other")

        return df

In [17]:
class RemoveStr(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        
        turbo_mask = df['Engine volume'].str.contains('Turbo', na=False)
        df['Engine volume'] = df['Engine volume'].str.extract(r'(\d\.?\d*)')[0].astype(float)
        df.loc[turbo_mask, 'Engine volume'] *= 1.5
        
        df['Mileage'] = df['Mileage'].str.replace('km', '', regex=False).astype(float)

        return df


In [18]:
# Filtering

class FilteringFeatureEngineering(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # Filtering

        # FeatureEngineering
        df['car_age'] = 2026 - df['Prod. year']
        df['Engine Displacement'] = df['Engine volume'] * df['Cylinders']
        df['Miles per year'] = df['Mileage'] / df['car_age'].replace(0, 1)

        # Conver to float
        df = df.astype({'Prod. year': float, 
                'Engine volume': float, 'Mileage': float, 
                'Cylinders': float, 'car_age': float, 
                'Engine Displacement': float, 'Miles per year': float})
        
        return df


In [19]:
class ModelCanonicalizer(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=10):
        self.min_count = min_count

    def _normalize_text(self, s):
        s = str(s).strip().lower()
        s = s.replace("–", "-").replace("—", "-")
        s = re.sub(r"[^a-z0-9\s\-]", "", s)
        s = re.sub(r"\s+", " ", s).strip()
        return s

    def _make_model_key(self, s):
        s = self._normalize_text(s)
        s = s.replace("-", "")
        s = s.replace(" ", "")
        return s

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        supported_models = [
            "Camry", "Corolla", "RAV4", "Highlander", "4Runner", "Tacoma", "Tundra",
            "Prius", "Prius C", "Prius V", "Sienna", "Sequoia", "Land Cruiser",
            "Land Cruiser Prado", "Crown", "Avalon", "C-HR", "Aqua", "Vitz", "Ist", "VOXY",
            "Elantra", "Sonata", "Tucson", "Santa Fe", "Palisade", "Kona", "Venue",
            "Santa Cruz", "Ioniq 5", "Ioniq 6", "Accent", "Genesis", "Grandeur", "Veloster", "i30", "H1",
            "Altima", "Sentra", "Versa", "Rogue", "Murano", "Pathfinder", "Armada",
            "Frontier", "Leaf", "Kicks", "Z", "Juke", "Tiida", "Note", "March", "X-Trail", "Xterra", "Serena", "Skyline",
            "K4", "K5", "Forte", "Soul", "Seltos", "Sportage", "Sorento", "Telluride", "Carnival", "EV6", "EV9", "Optima", "Rio", "Picanto", "Cerato",
            "Accord", "Civic", "CR-V", "HR-V", "Pilot", "Odyssey", "Passport", "Ridgeline", "Prologue", "Fit", "Insight", "Elysion",
            "RX 350", "RX 450", "NX 350", "ES 350", "ES 300", "GX 550", "GX 460", "GX 470",
            "LX 600", "LS 500", "LS 460", "IS 350", "IS 250", "RC 350", "TX 350", "UX 300h", "CT 200h", "HS 250h",
            "Terrain", "Acadia", "Yukon", "Yukon XL", "Canyon", "Sierra 1500", "Sierra HD", "Hummer EV",
            "Trax", "Trailblazer", "Equinox", "Blazer", "Traverse", "Tahoe", "Suburban", "Colorado", "Silverado 1500",
            "Malibu", "Corvette", "Cruze", "Captiva", "Volt", "Orlando", "Spark", "Aveo", "Impala", "Camaro",
            "Maverick", "Ranger", "F-150", "Mustang", "Escape", "Bronco Sport", "Bronco", "Explorer", "Expedition",
            "Transit", "Fusion", "Focus", "Fiesta", "Taurus", "Transit Connect",
            "Model 3", "Model Y", "Model S", "Model X", "Cybertruck",
            "2 Series", "3 Series", "4 Series", "5 Series", "7 Series", "X1", "X3", "X5", "X6", "X7", "i4", "iX",
            "C-Class", "E-Class", "S-Class", "CLA", "CLS", "GLA", "GLC", "GLE", "GLS", "GL", "ML", "EQB", "EQE", "G-Class", "Sprinter", "Vito",
            "Mirage", "Outlander", "Outlander Sport", "Eclipse Cross", "Pajero", "Pajero iO", "Airtrek", "Colt",
            "Range Rover", "Range Rover Sport", "Range Rover Velar", "Range Rover Evoque", "Discovery", "Defender",
            "Compass", "Cherokee", "Grand Cherokee", "Wrangler", "Gladiator", "Wagoneer", "Grand Wagoneer",
            "Hornet", "Durango", "Charger", "Challenger", "Caliber",
            "1500", "2500", "3500", "ProMaster",
            "Jetta", "Taos", "Tiguan", "Atlas", "Atlas Cross Sport", "Golf GTI", "Golf R", "Golf", "ID.4", "Passat", "CC",
            "A3", "A4", "A5", "A6", "A7", "Q3", "Q5", "Q7", "Q8", "e-tron", "Q4 e-tron",
            "Mazda3", "Mazda6", "CX-30", "CX-5", "CX-9", "CX-50", "CX-70", "CX-90", "MX-5 Miata", "MPV", "Demio",
            "Q50", "QX50", "QX55", "QX60", "QX80",
            "CT4", "CT5", "XT4", "XT5", "XT6", "Escalade", "Lyriq", "Optiq",
            "Impreza", "Legacy", "WRX", "Crosstrek", "Forester", "Outback", "Ascent", "BRZ", "Solterra", "XV"
        ]

        model_aliases = {
            "rav4": "RAV4", "rav 4": "RAV4", "chr": "C-HR",
            "camryse": "Camry", "camryxle": "Camry",
            "priusc": "Prius C", "priusv": "Prius V", "landcruiserprado": "Land Cruiser Prado",
            "h1": "H1", "i30": "i30",
            "xtrail": "X-Trail", "xterra": "Xterra", "xterraa": "Xterra",
            "gx460": "GX 460", "gx470": "GX 470", "rx450": "RX 450",
            "es300": "ES 300", "ls460": "LS 460", "is250": "IS 250",
            "ct200h": "CT 200h", "hs250h": "HS 250h",
            "cruzelt": "Cruze", "silverado1500": "Silverado 1500",
            "f150": "F-150", "transitconnect": "Transit Connect",
            "model3": "Model 3", "modely": "Model Y", "models": "Model S", "modelx": "Model X",
            "328": "3 Series", "320": "3 Series", "318": "3 Series", "330": "3 Series", "335": "3 Series",
            "525": "5 Series", "528": "5 Series", "530": "5 Series", "535": "5 Series", "550": "5 Series",
            "c300": "C-Class", "c250": "C-Class", "c200": "C-Class", "c180": "C-Class",
            "e350": "E-Class", "e300": "E-Class", "e320": "E-Class", "e200": "E-Class",
            "s550": "S-Class", "gla250": "GLA", "gle350": "GLE", "gl450": "GL", "ml350": "ML",
            "cla250": "CLA", "cls550": "CLS",
            "pajeroio": "Pajero iO",
            "golfgti": "Golf GTI", "golfr": "Golf R",
            "mazda6": "Mazda6", "cx9": "CX-9", "xv": "XV"
        }

        model_lookup = {self._make_model_key(model): model for model in supported_models}

        def canonicalize_model(raw_model):
            if pd.isna(raw_model):
                return None

            raw_key = self._make_model_key(raw_model)

            if raw_key in model_aliases:
                return model_aliases[raw_key]

            if raw_key in model_lookup:
                return model_lookup[raw_key]

            for model_key, canonical_model in sorted(model_lookup.items(), key=lambda x: len(x[0]), reverse=True):
                if raw_key.startswith(model_key):
                    return canonical_model

            return None

        df["Model"] = df["Model"].apply(canonicalize_model)

        df = df[df["Model"].notna()].copy()

        model_counts = df["Model"].value_counts()
        rare_models = model_counts[model_counts < self.min_count].index
        df["Model"] = df["Model"].replace(rare_models, "other")

        return df


In [20]:
class ModelTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=10):
        self.alpha = alpha

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y, index=X.index, name="Price")

        stats = pd.DataFrame({
            "Model": X["Model"],
            "Price": y
        })

        model_counts = stats.groupby("Model")["Price"].count()
        model_means = stats.groupby("Model")["Price"].mean()

        self.global_mean_ = y.mean()

        self.model_te_ = (
            (model_means * model_counts) + (self.global_mean_ * self.alpha)
        ) / (model_counts + self.alpha)

        return self

    def transform(self, X):
        X = X.copy()

        X["Model_te"] = X["Model"].map(self.model_te_).fillna(self.global_mean_)
        X = X.drop(columns=["Model"])

        return X[["Model_te"]]


In [21]:
categorical_features = [
    "Manufacturer",
    "Color",
    "Category",
    "Fuel type",
    "Drive wheels",
    "Gear box type"
]

model_feature = ["Model"]

numeric_features = [
    "Prod. year",
    "Engine volume",
    "Mileage",
    "Cylinders",
    "car_age",
    "Engine Displacement",
    "Miles per year"
]

preprocessor = ColumnTransformer([
    ("model_te", ModelTargetEncoder(alpha=10), model_feature),
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=1000,
        random_state=42,
        n_jobs=-1
    ))
])


# -----------------------------
# 1) Filter target first
# -----------------------------
df = df[(df["Price"] >= 500) & (df["Price"] < 150000)].copy()


# -----------------------------
# 2) Clean raw dataframe BEFORE train/test split
#    These steps can remove rows, so keep them outside the sklearn model pipeline.
# -----------------------------
cleaning_pipeline = Pipeline([
    ("mapping", Mapping()),
    ("remove_str", RemoveStr()),
    ("feature_engineering", FilteringFeatureEngineering()),
    ("make_cleaner", MakeCanonicalizer()),
    ("model_cleaner", ModelCanonicalizer())
])

df_clean = cleaning_pipeline.fit_transform(df)


# -----------------------------
# 3) Filter rows AFTER Mapping + RemoveStr + FeatureEngineering
# -----------------------------
df_clean = df_clean[df_clean["Cylinders"].isin([2, 3, 4, 5, 6, 8, 12])].copy()
df_clean = df_clean[(df_clean["Mileage"] >= 1000) & (df_clean["Mileage"] <= 200000)].copy()
df_clean = df_clean[df_clean["Engine volume"].isin([2, 2.5, 1.8, 1.5])].copy()

df_clean = df_clean.dropna(subset=[
    "Manufacturer",
    "Model",
    "Color",
    "Category",
    "Fuel type",
    "Drive wheels",
    "Gear box type",
    "Prod. year",
    "Engine volume",
    "Mileage",
    "Cylinders",
    "Price"
]).copy()


# -----------------------------
# 4) Train/test split
# -----------------------------
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42
)

X_train = train_df.drop(columns=["Price"])
y_train = train_df["Price"]

X_test = test_df.drop(columns=["Price"])
y_test = test_df["Price"]


# -----------------------------
# 5) Fit model
# -----------------------------
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


MAE: 4390.6682385627855
R2: 0.7236432656691524


In [22]:
import joblib
joblib.dump(model_pipeline, "../car_price_pipeline.pkl")

['../car_price_pipeline.pkl']